# 03 Baseline Model Comparison

This notebook runs the individual grounding baselines using the benchmark produced in Notebook 02:

- Grounding DINO
- OWL ViT
- GPT Vision

The task parser shortens complex commands for the local open vocabulary detectors. Ground truth is always loaded from `examples/ground_truth_boxes.csv`.

Run Notebook 02 first. Notebook 04 generates the GPT-guided OWL-ViT result. Notebook 05 combines all four methods.


In [ ]:
# Colab dependency setup.
# Do not reinstall Pillow inside an active runtime. A forced reinstall can leave
# ImageText.py and PIL._typing from different Pillow versions.

%pip -q install --upgrade-strategy only-if-needed pandas numpy matplotlib tqdm openai
%pip -q install --upgrade-strategy only-if-needed transformers accelerate safetensors

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Drive is already mounted or this is not a Colab runtime.')

from pathlib import Path
import os
import sys
import json
import re
import time
import subprocess
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from IPython.display import display

print('Pillow version:', Image.__version__)


In [ ]:
# Project path and run configuration.

PROJECT_ROOT = Path('/content/drive/MyDrive/autonomous-delivery-robot/modules/perception/visual_grounding')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_ROOT}. Update PROJECT_ROOT above.')

# Model switches
# Complete three-baseline rerun. Notebook 04 generates GPT-guided OWL-ViT.
RUN_GROUNDING_DINO = True
RUN_OWL_VIT = True
RUN_GPT_VISION = True

# Old result files were produced against an obsolete Day 3 CSV and old image IDs.
# Keep this True the first time after rebuilding ground truth, so stale outputs cannot contaminate metrics.
CLEAR_OLD_MODEL_OUTPUTS_BEFORE_RUN = True

# GPT Vision
GPT_VISION_MODEL = 'gpt-5'
GPT_VISION_DETAIL = 'high'
GPT_VISION_MAX_ROWS = None
GPT_VISION_DEBUG_FAILURES = 10

# Run limits
MAX_ROWS = None
MAX_SCREENSHOTS_PER_MODEL = 100
SCREENSHOT_IMAGE_FORMAT = 'png'  # PNG keeps thin boxes sharper than JPEG.

# Evaluation thresholds
SUCCESS_IOU_THRESHOLD = 0.25
STRICT_SUCCESS_IOU_THRESHOLD = 0.50
WEAK_OVERLAP_IOU_THRESHOLD = 0.10

# Local model thresholds
GROUNDING_DINO_RETRY_THRESHOLDS = [0.08, 0.05, 0.02, 0.005, 0.0]
OWL_VIT_RETRY_THRESHOLDS = [0.05, 0.02, 0.01, 0.005, 0.0]

assert all(isinstance(t, (int, float)) for t in GROUNDING_DINO_RETRY_THRESHOLDS)
assert all(isinstance(t, (int, float)) for t in OWL_VIT_RETRY_THRESHOLDS)

# Candidate prompt settings
CANDIDATE_LABEL_MODE = 'target_first'
MAX_PRED_BOX_AREA_RATIO = 0.90
MIN_PRED_BOX_AREA_RATIO = 0.0002
FULL_IMAGE_REJECT_RATIO = 0.97
LABEL_PRIORITY_BONUS = 0.04
PRED_BOX_AREA_PENALTY = 0.08


# Task parser settings
# The default parser is rule based so the notebook runs without API keys.
# Set TASK_PARSER_MODE = 'openai_text' later if you want to replace this with an LLM parser.
ENABLE_TASK_PARSER = True
TASK_PARSER_MODE = 'rule_based'
USE_PARSED_PROMPTS_FOR_LOCAL_MODELS = True
MAX_GROUNDING_CANDIDATES = 12


# Paths
BENCHMARK_PATH = PROJECT_ROOT / 'examples/grounding_benchmark_from_ground_truth.csv'
GT_COORDINATE_CSV_PATH = PROJECT_ROOT / 'examples/ground_truth_boxes.csv'
PARSED_BENCHMARK_PATH = PROJECT_ROOT / 'examples/grounding_benchmark_with_task_parser.csv'
TASK_PARSER_AUDIT_PATH = PROJECT_ROOT / 'results/model_comparison/task_parser_audit.csv'

OUTPUT_DIR = PROJECT_ROOT / 'examples/outputs'
SCREENSHOT_DIR = PROJECT_ROOT / 'results/screenshots'
MODEL_COMPARISON_DIR = PROJECT_ROOT / 'results/model_comparison'
EXTERNAL_DIR = PROJECT_ROOT / 'external'

MODEL_RESULT_PATHS = {
    'grounding_dino': OUTPUT_DIR / 'grounding_dino_results.csv',
    'owlvit': OUTPUT_DIR / 'owlvit_results.csv',
    'gpt_vision': OUTPUT_DIR / 'gpt_vision_results.csv',
}

MODEL_RUN_FLAGS = {
    'grounding_dino': RUN_GROUNDING_DINO,
    'owlvit': RUN_OWL_VIT,
    'gpt_vision': RUN_GPT_VISION,
}

for p in [OUTPUT_DIR, SCREENSHOT_DIR, MODEL_COMPARISON_DIR, EXTERNAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def clear_old_outputs_if_requested():
    """Clear only models selected for rerun.

    Results for skipped models are preserved so Notebook 05 can combine newly
    generated DINO outputs with existing OWL ViT/GPT Vision outputs.
    The nested Notebook 04 folder is never touched.
    """
    if not CLEAR_OLD_MODEL_OUTPUTS_BEFORE_RUN:
        print('Keeping existing model outputs. Make sure they match the current benchmark.')
        return

    cleared_models = []
    for model_key, should_run in MODEL_RUN_FLAGS.items():
        if not should_run:
            continue

        MODEL_RESULT_PATHS[model_key].unlink(missing_ok=True)

        for folder_name in [model_key]:
            folder = SCREENSHOT_DIR / folder_name
            if folder.exists():
                shutil.rmtree(folder)
            folder.mkdir(parents=True, exist_ok=True)

        for path in [
            MODEL_COMPARISON_DIR / f'{model_key}_metrics.csv',
            MODEL_COMPARISON_DIR / f'{model_key}_screenshot_audit.csv',
        ]:
            path.unlink(missing_ok=True)

        cleared_models.append(model_key)

    # These files are regenerated from every currently available baseline result.
    combined_folder = SCREENSHOT_DIR / 'combined_models'
    if combined_folder.exists():
        shutil.rmtree(combined_folder)
    combined_folder.mkdir(parents=True, exist_ok=True)

    for name in [
        'all_model_metrics.csv',
        'mean_iou_by_model.png',
        'success_rate_by_model.png',
        'bbox_quality_counts_by_model.csv',
        'bbox_quality_counts_by_model.png',
        'metrics_by_target_object.csv',
        'success_rate_by_target_object.png',
        'notebook03_output_contract.csv',
    ]:
        (MODEL_COMPARISON_DIR / name).unlink(missing_ok=True)

    print('Cleared outputs only for models selected to rerun:', cleared_models)
    print('Preserved skipped-model CSVs and examples/outputs/gpt_guided_owlvit/.')


clear_old_outputs_if_requested()

DEVICE = 'cuda' if subprocess.run(
    'nvidia-smi',
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
).returncode == 0 else 'cpu'

print('Device:', DEVICE)
print('Project root:', PROJECT_ROOT)
print('Benchmark CSV:', BENCHMARK_PATH)
print('Ground truth CSV:', GT_COORDINATE_CSV_PATH)
print('Parsed benchmark CSV:', PARSED_BENCHMARK_PATH)
print('This notebook does not use examples/day3_selected_frames.csv')



In [ ]:
def clean_path(value):
    return str(value).replace('\\', '/').replace('\\', '/').strip()


def find_coordinate_columns(df):
    options = [
        ('gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max'),
        ('x_min', 'y_min', 'x_max', 'y_max'),
        ('bbox_x_min', 'bbox_y_min', 'bbox_x_max', 'bbox_y_max'),
        ('ground_truth_x_min', 'ground_truth_y_min', 'ground_truth_x_max', 'ground_truth_y_max'),
    ]
    for cols in options:
        if all(c in df.columns for c in cols):
            return cols
    return None


def standardize_gt_coordinates(df, source_name):
    df = df.copy()
    if 'image_id' not in df.columns:
        raise ValueError(f'{source_name} is missing image_id.')

    cols = find_coordinate_columns(df)
    if cols is None:
        raise ValueError(f'{source_name} does not contain ground truth coordinate columns.')

    out = df[['image_id']].copy()
    out['image_id'] = out['image_id'].astype(str)
    out['gt_x_min'] = pd.to_numeric(df[cols[0]], errors='coerce')
    out['gt_y_min'] = pd.to_numeric(df[cols[1]], errors='coerce')
    out['gt_x_max'] = pd.to_numeric(df[cols[2]], errors='coerce')
    out['gt_y_max'] = pd.to_numeric(df[cols[3]], errors='coerce')
    out['ground_truth_source'] = source_name
    out = out.dropna(subset=['gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max'])
    out = out.drop_duplicates('image_id', keep='last')
    return out


def relative_project_path(path):
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_ROOT)).replace('\\', '/')
    except Exception:
        return str(path).replace('\\', '/')


def resolve_raw_image_path(image_id, image_path_value):
    image_id = str(image_id)

    candidates = []
    if pd.notna(image_path_value):
        cleaned = clean_path(image_path_value)
        if cleaned:
            p = Path(cleaned)
            candidates.append(PROJECT_ROOT / p if not p.is_absolute() else p)

    raw_dir = PROJECT_ROOT / 'examples' / 'residence_images'
    for ext in ['.jpg', '.jpeg', '.png', '.webp']:
        candidates.append(raw_dir / f'{image_id}{ext}')

    candidates.extend(sorted(raw_dir.glob(f'{image_id}.*')))

    for p in candidates:
        if p.exists() and p.is_file():
            return p

    return None


def normalize_box_xyxy(box, image_size):
    if box is None:
        return None

    try:
        vals = [float(v) for v in box]
    except Exception:
        return None

    if len(vals) != 4 or any(np.isnan(vals)):
        return None

    w, h = image_size
    x1, y1, x2, y2 = vals

    # Handle normalized boxes if any model returns 0 to 1 coordinates.
    if max(abs(x1), abs(y1), abs(x2), abs(y2)) <= 1.5:
        x1, x2 = x1 * w, x2 * w
        y1, y2 = y1 * h, y2 * h

    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1

    x1 = min(max(x1, 0.0), float(w - 1))
    x2 = min(max(x2, 0.0), float(w - 1))
    y1 = min(max(y1, 0.0), float(h - 1))
    y2 = min(max(y2, 0.0), float(h - 1))

    if (x2 - x1) <= 1 or (y2 - y1) <= 1:
        return None

    return [x1, y1, x2, y2]


def load_benchmark():
    if not BENCHMARK_PATH.exists():
        raise FileNotFoundError(f'Benchmark CSV not found: {BENCHMARK_PATH}')
    if not GT_COORDINATE_CSV_PATH.exists():
        raise FileNotFoundError(f'Ground truth coordinate CSV not found: {GT_COORDINATE_CSV_PATH}')

    benchmark = pd.read_csv(BENCHMARK_PATH)
    gt_raw = pd.read_csv(GT_COORDINATE_CSV_PATH)

    if 'image_id' not in benchmark.columns:
        raise ValueError('grounding_benchmark_from_ground_truth.csv must contain image_id.')
    if 'image_path' not in benchmark.columns:
        raise ValueError('grounding_benchmark_from_ground_truth.csv must contain image_path.')

    benchmark['image_id'] = benchmark['image_id'].astype(str)
    benchmark['image_path'] = benchmark['image_path'].map(clean_path)

    # Ground truth CSV is the final source of approved evaluation rows.
    final_gt = standardize_gt_coordinates(gt_raw, 'ground_truth_boxes.csv')

    original_benchmark_count = len(benchmark)
    original_gt_count = len(final_gt)

    # Remove any stale coordinates in the benchmark CSV and replace them from ground_truth_boxes.csv.
    stale_gt_cols = [
        'gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max',
        'x_min', 'y_min', 'x_max', 'y_max',
        'bbox_x_min', 'bbox_y_min', 'bbox_x_max', 'bbox_y_max',
        'ground_truth_x_min', 'ground_truth_y_min', 'ground_truth_x_max', 'ground_truth_y_max',
        'ground_truth_source',
    ]
    benchmark = benchmark.drop(columns=[c for c in stale_gt_cols if c in benchmark.columns], errors='ignore')

    benchmark_ids = set(benchmark['image_id'].astype(str))
    gt_ids = set(final_gt['image_id'].astype(str))

    rows_without_gt = sorted(benchmark_ids - gt_ids)
    gt_without_benchmark = sorted(gt_ids - benchmark_ids)

    # Inner join means only images with both a benchmark prompt row and final ground truth coordinates are evaluated.
    benchmark = benchmark.merge(final_gt, on='image_id', how='inner')

    resolved_paths = []
    missing_raw_rows = []

    for _, row in benchmark.iterrows():
        raw_path = resolve_raw_image_path(row['image_id'], row.get('image_path', ''))
        if raw_path is None:
            resolved_paths.append('')
            missing_raw_rows.append({'image_id': row['image_id'], 'image_path': row.get('image_path', '')})
        else:
            resolved_paths.append(relative_project_path(raw_path))

    benchmark['image_path'] = resolved_paths

    if missing_raw_rows:
        missing_raw_df = pd.DataFrame(missing_raw_rows)
        print(f'Skipped rows with missing raw residence images: {len(missing_raw_df)}')
        display(missing_raw_df.head(20))

    benchmark = benchmark[benchmark['image_path'].astype(str).str.len() > 0].copy()

    # Ensure numeric coordinate columns.
    for col in ['gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max']:
        benchmark[col] = pd.to_numeric(benchmark[col], errors='coerce')

    before_valid_gt = len(benchmark)
    benchmark = benchmark.dropna(subset=['gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max']).copy()
    dropped_invalid_gt = before_valid_gt - len(benchmark)

    # Keep rows stable and reproducible.
    benchmark = benchmark.sort_values('image_id').reset_index(drop=True)

    if MAX_ROWS is not None:
        benchmark = benchmark.head(int(MAX_ROWS)).copy()

    print('Benchmark filtering summary')
    print(f'Benchmark rows loaded: {original_benchmark_count}')
    print(f'Approved ground truth rows loaded: {original_gt_count}')
    print(f'Benchmark rows skipped because no approved ground truth exists: {len(rows_without_gt)}')
    print(f'Approved ground truth rows not used because no benchmark prompt exists: {len(gt_without_benchmark)}')
    print(f'Rows dropped because ground truth coordinates were invalid: {dropped_invalid_gt}')
    print(f'Final evaluation rows: {len(benchmark)}')

    if rows_without_gt:
        print('First skipped benchmark image IDs without approved ground truth:')
        print(rows_without_gt[:20])
    if gt_without_benchmark:
        print('First approved ground truth image IDs without benchmark prompt rows:')
        print(gt_without_benchmark[:20])

    return benchmark



# ---------------------------------------------------------------------
# Task parser
# ---------------------------------------------------------------------
# Converts long commands or filename descriptions into a short model prompt.
# Example:
#   command: Pick up the package beside the chair near the elevator.
#   parsed_target_object: package
#   parsed_location_hint: beside chair near elevator
#   parsed_grounding_prompt: package beside chair
#   parsed_action: pickup

STOP_WORDS = {
    'the', 'a', 'an', 'this', 'that', 'these', 'those', 'please', 'can', 'you',
    'me', 'to', 'from', 'there', 'here', 'visible', 'target', 'object',
}

ACTION_PATTERNS = [
    ('pickup', [r'\bpick\s*up\b', r'\bgrab\b', r'\bcollect\b', r'\btake\b']),
    ('dropoff', [r'\bdrop\s*off\b', r'\bdeliver\b', r'\bplace\b', r'\bleave\b', r'\bput\b']),
    ('find', [r'\bfind\b', r'\blocate\b', r'\blook\s*for\b', r'\bidentify\b', r'\bdetect\b']),
    ('press', [r'\bpress\b', r'\bpush\b', r'\btap\b']),
    ('open', [r'\bopen\b']),
]

OBJECT_RULES = [
    ('elevator button panel', ['elevator button panel', 'button panel', 'elevator button', 'buttons', 'button']),
    ('package', ['delivery package', 'cardboard box', 'parcel', 'package', 'box']),
    ('person', ['person', 'human', 'man', 'woman', 'girl', 'boy']),
    ('door', ['apartment door', 'hallway door', 'elevator door', 'door']),
    ('chair', ['chair', 'seat']),
    ('table', ['table', 'desk']),
    ('couch', ['couch', 'sofa']),
    ('bag', ['backpack', 'bag']),
    ('shoes', ['shoes', 'shoe']),
    ('plant', ['plant', 'flower']),
    ('sign', ['sign', 'label', 'poster']),
    ('water dispenser', ['water dispenser', 'dispenser']),
    ('elevator', ['elevator', 'lift']),
    ('lobby', ['lobby']),
    ('lounge', ['lounge']),
    ('hallway', ['hallway', 'corridor']),
]

OBJECT_SYNONYMS = {
    'package': ['package', 'box', 'cardboard box', 'delivery package', 'parcel'],
    'person': ['person', 'human'],
    'door': ['door', 'apartment door', 'hallway door'],
    'elevator button panel': ['elevator button panel', 'button panel', 'elevator button', 'button'],
    'chair': ['chair'],
    'table': ['table', 'desk'],
    'couch': ['couch', 'sofa'],
    'bag': ['bag', 'backpack'],
    'shoes': ['shoes', 'shoe'],
    'plant': ['plant'],
    'sign': ['sign', 'label'],
    'water dispenser': ['water dispenser', 'dispenser'],
    'elevator': ['elevator', 'lift'],
    'lobby': ['lobby'],
    'lounge': ['lounge'],
    'hallway': ['hallway', 'corridor'],
}

RELATION_WORDS = [
    'beside', 'next to', 'near', 'by', 'beside the', 'next to the', 'near the', 'by the',
    'in front of', 'behind', 'under', 'on top of', 'on', 'above', 'below', 'inside', 'outside',
    'between', 'left of', 'right of', 'at', 'around', 'close to', 'against', 'underneath'
]

LOCATION_ANCHOR_WORDS = [
    'chair', 'table', 'desk', 'elevator', 'door', 'wall', 'floor', 'lobby', 'lounge',
    'hallway', 'corridor', 'couch', 'sofa', 'plant', 'shoes', 'books', 'box', 'package',
    'person', 'bag', 'sign', 'water dispenser'
]

FRAME_TOKEN_RE = re.compile(r'\b(vid|res|frame|image|img|jpg|jpeg|png|mp4|ground|truth|annotated)\b|\b\d+\b')


def safe_text(value):
    value = '' if value is None else str(value)
    if value.strip().lower() in {'nan', 'none', 'null'}:
        return ''
    return value.strip()


def normalize_task_text(text):
    text = safe_text(text).lower()
    text = text.replace('_', ' ').replace('-', ' ').replace('/', ' ')
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def unique_keep_order(values):
    out = []
    seen = set()
    for value in values:
        value = safe_text(value)
        if not value:
            continue
        key = value.lower()
        if key not in seen:
            out.append(value)
            seen.add(key)
    return out


def remove_frame_tokens(text):
    text = FRAME_TOKEN_RE.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def choose_command_text(row):
    # Use the command if it exists, otherwise use the filename description or original grounding prompt.
    for col in ['command', 'filename_description', 'grounding_prompt', 'annotation_id', 'image_id']:
        if col in row:
            value = safe_text(row.get(col, ''))
            if value:
                return value
    return ''


def infer_action_from_text(text):
    text_l = normalize_task_text(text)
    for action, patterns in ACTION_PATTERNS:
        for pattern in patterns:
            if re.search(pattern, text_l):
                return action
    return 'find'


def strip_action_words(text):
    text_l = normalize_task_text(text)
    for _, patterns in ACTION_PATTERNS:
        for pattern in patterns:
            text_l = re.sub(pattern, ' ', text_l)
    text_l = re.sub(r'\b(find|locate|object|target|visible|please|the|a|an)\b', ' ', text_l)
    text_l = re.sub(r'\s+', ' ', text_l).strip()
    return text_l


def infer_target_from_text(text, fallback='object'):
    text_l = normalize_task_text(text)
    best_target = ''
    best_pos = 10**9
    for target, keywords in OBJECT_RULES:
        for keyword in keywords:
            pattern = r'\b' + re.escape(keyword) + r'\b'
            match = re.search(pattern, text_l)
            if match and match.start() < best_pos:
                best_target = target
                best_pos = match.start()
    if best_target:
        return best_target
    fallback = safe_text(fallback).lower()
    if fallback and fallback not in {'nan', 'none', 'object', 'target'}:
        return fallback
    words = [w for w in remove_frame_tokens(text_l).split() if w not in STOP_WORDS]
    return words[0] if words else 'object'


def remove_target_phrase(text, target_object):
    out = normalize_task_text(text)
    for synonym in OBJECT_SYNONYMS.get(target_object, [target_object]):
        out = re.sub(r'\b' + re.escape(synonym) + r'\b', ' ', out)
    out = re.sub(r'\s+', ' ', out).strip()
    return out


def normalize_location_hint(text):
    text_l = normalize_task_text(text)
    # remove action and frame/noise tokens, but keep relation words and anchors
    text_l = strip_action_words(text_l)
    text_l = remove_frame_tokens(text_l)
    text_l = re.sub(r'\b(the|a|an|this|that|nearby)\b', ' ', text_l)
    text_l = re.sub(r'\s+', ' ', text_l).strip()
    return text_l


def extract_location_hint(command_text, target_object):
    text_l = normalize_task_text(command_text)
    cleaned = strip_action_words(text_l)

    # Prefer the phrase that begins at the first spatial relation after the target.
    target_positions = []
    for synonym in OBJECT_SYNONYMS.get(target_object, [target_object]):
        m = re.search(r'\b' + re.escape(synonym) + r'\b', cleaned)
        if m:
            target_positions.append(m.end())
    start_search = min(target_positions) if target_positions else 0
    after_target = cleaned[start_search:]

    relation_hits = []
    for rel in RELATION_WORDS:
        m = re.search(r'\b' + re.escape(rel) + r'\b', after_target)
        if m:
            relation_hits.append((m.start(), rel))
    if relation_hits:
        relation_start = min(relation_hits)[0]
        location = after_target[relation_start:]
    else:
        location = remove_target_phrase(cleaned, target_object)

    return normalize_location_hint(location)


def choose_primary_anchor(location_hint):
    location = normalize_task_text(location_hint)
    best_anchor = ''
    best_pos = 10**9
    for anchor in LOCATION_ANCHOR_WORDS:
        m = re.search(r'\b' + re.escape(anchor) + r'\b', location)
        if m and m.start() < best_pos:
            best_anchor = anchor
            best_pos = m.start()
    return best_anchor


def choose_primary_relation(location_hint):
    location = normalize_task_text(location_hint)
    for rel in ['beside', 'next to', 'near', 'by', 'in front of', 'behind', 'under', 'on', 'inside', 'outside', 'between', 'at']:
        if re.search(r'\b' + re.escape(rel) + r'\b', location):
            return rel
    return ''


def build_grounding_prompt(target_object, location_hint):
    target = safe_text(target_object).lower() or 'object'
    location = normalize_task_text(location_hint)
    relation = choose_primary_relation(location)
    anchor = choose_primary_anchor(location)

    if relation and anchor and anchor != target:
        return f'{target} {relation} {anchor}'.strip()
    if location:
        words = [w for w in location.split() if w not in STOP_WORDS]
        short_location = ' '.join(words[:4])
        if short_location:
            return f'{target} {short_location}'.strip()
    return target


def parser_candidate_prompts(target_object, grounding_prompt, location_hint):
    target = safe_text(target_object).lower() or 'object'
    prompt = safe_text(grounding_prompt).lower()
    location = normalize_task_text(location_hint)
    relation = choose_primary_relation(location)
    anchor = choose_primary_anchor(location)

    candidates = []
    candidates.extend(OBJECT_SYNONYMS.get(target, [target]))
    if prompt:
        candidates.insert(0, prompt)
    if relation and anchor:
        candidates.append(f'{target} {relation} {anchor}')
    if anchor:
        candidates.append(f'{target} near {anchor}')
        candidates.append(f'{target} by {anchor}')
    return unique_keep_order([c.lower().rstrip('.') for c in candidates if safe_text(c)])


def parse_task_row_rule_based(row):
    source_command = choose_command_text(row)
    fallback_target = row.get('target_object', 'object')
    target = infer_target_from_text(source_command, fallback=fallback_target)
    action = infer_action_from_text(source_command)
    location = extract_location_hint(source_command, target)
    prompt = build_grounding_prompt(target, location)
    candidates = parser_candidate_prompts(target, prompt, location)

    confidence = 0.75
    if target != 'object':
        confidence += 0.10
    if location:
        confidence += 0.05
    confidence = min(confidence, 0.95)

    return {
        'source_command': source_command,
        'parsed_action': action,
        'parsed_target_object': target,
        'parsed_location_hint': location,
        'parsed_grounding_prompt': prompt,
        'model_grounding_prompt': prompt,
        'parser_candidate_prompts': ' | '.join(candidates),
        'parser_source': 'rule_based_text_parser',
        'parser_confidence': confidence,
    }


def apply_task_parser(benchmark_df):
    df = benchmark_df.copy()

    if not ENABLE_TASK_PARSER:
        df['source_command'] = df.get('command', df.get('grounding_prompt', '')).astype(str)
        df['parsed_action'] = 'find'
        df['parsed_target_object'] = df.get('target_object', 'object')
        df['parsed_location_hint'] = ''
        df['parsed_grounding_prompt'] = df.get('grounding_prompt', df.get('target_object', 'object'))
        df['model_grounding_prompt'] = df['parsed_grounding_prompt']
        df['parser_candidate_prompts'] = ''
        df['parser_source'] = 'parser_disabled'
        df['parser_confidence'] = np.nan
        return df

    parsed_rows = []
    for row in df.to_dict('records'):
        # Default to local parser. Optional LLM parser can be added later without changing model code.
        parsed_rows.append(parse_task_row_rule_based(row))

    parsed_df = pd.DataFrame(parsed_rows)
    df = pd.concat([df.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)

    # Use parsed fields in the model-facing columns, but preserve originals for audit.
    df['original_grounding_prompt'] = df.get('grounding_prompt', '')
    df['original_target_object'] = df.get('target_object', '')
    df['grounding_prompt'] = df['model_grounding_prompt']
    df['target_object'] = df['parsed_target_object']
    df['target_label'] = df['parsed_target_object']

    if 'target_type' not in df.columns:
        df['target_type'] = 'object'
    df.loc[df['parsed_target_object'].isin(['person']), 'target_type'] = 'person'
    df.loc[df['parsed_target_object'].isin(['lobby', 'lounge', 'hallway', 'elevator']), 'target_type'] = 'place'
    df.loc[~df['parsed_target_object'].isin(['person', 'lobby', 'lounge', 'hallway', 'elevator']), 'target_type'] = 'object'

    parser_cols = [
        'image_id', 'source_command', 'parsed_action', 'parsed_target_object',
        'parsed_location_hint', 'parsed_grounding_prompt', 'model_grounding_prompt',
        'parser_candidate_prompts', 'parser_source', 'parser_confidence',
        'original_grounding_prompt', 'original_target_object',
    ]
    PARSED_BENCHMARK_PATH.parent.mkdir(parents=True, exist_ok=True)
    TASK_PARSER_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(PARSED_BENCHMARK_PATH, index=False)
    df[parser_cols].to_csv(TASK_PARSER_AUDIT_PATH, index=False)

    print('Task parser complete')
    print(f'Saved parsed benchmark: {PARSED_BENCHMARK_PATH}')
    print(f'Saved parser audit: {TASK_PARSER_AUDIT_PATH}')
    print('Parser source counts:')
    display(df['parser_source'].value_counts(dropna=False).reset_index().rename(columns={'index': 'parser_source', 'parser_source': 'rows'}))

    return df

benchmark = load_benchmark()

if 'grounding_prompt' not in benchmark.columns:
    raise ValueError('Benchmark CSV is missing grounding_prompt. Run Notebook 02 to create examples/grounding_benchmark_from_ground_truth.csv.')
if 'target_object' not in benchmark.columns:
    benchmark['target_object'] = benchmark['grounding_prompt'].astype(str).str.split().str[0].fillna('object')
if 'target_type' not in benchmark.columns:
    benchmark['target_type'] = 'object'
if 'target_label' not in benchmark.columns:
    benchmark['target_label'] = benchmark['target_object']
if 'command' not in benchmark.columns:
    benchmark['command'] = benchmark['grounding_prompt'].map(lambda x: f'find {x}')

benchmark = apply_task_parser(benchmark)

print('Unique model prompts after parser:', benchmark['grounding_prompt'].nunique())
print('Ground truth sources:')
display(benchmark['ground_truth_source'].value_counts(dropna=False).reset_index().rename(columns={'index': 'source', 'ground_truth_source': 'rows'}))

preview_cols = [
    'image_id', 'image_path', 'source_command', 'parsed_action', 'parsed_target_object',
    'parsed_location_hint', 'model_grounding_prompt', 'gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max'
]
display(benchmark[[c for c in preview_cols if c in benchmark.columns]].head(40))


In [ ]:

def run_shell(cmd, cwd=None):
    print(cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=str(cwd) if cwd else None)


def clean_prompt(prompt):
    prompt = str(prompt).strip()
    if not prompt.endswith('.'):
        prompt += '.'
    return prompt


def unique_keep_order(values):
    out = []
    seen = set()
    for value in values:
        value = str(value).strip()
        if not value or value.lower() in {'nan', 'none'}:
            continue
        key = value.lower()
        if key not in seen:
            out.append(value)
            seen.add(key)
    return out


def simplified_prompt_text(prompt):
    text = str(prompt).strip()
    for prefix in [
        'Locate the ', 'Locate ', 'Find the ', 'Find ', 'Please locate the ',
        'please locate the ', 'locate the ', 'locate ', 'find the ', 'find ',
    ]:
        if text.startswith(prefix):
            text = text[len(prefix):]
            break
    return text.strip().rstrip('.')


def model_prompt_text(row):
    if USE_PARSED_PROMPTS_FOR_LOCAL_MODELS:
        for col in ['model_grounding_prompt', 'parsed_grounding_prompt', 'grounding_prompt']:
            value = str(row.get(col, '')).strip()
            if value and value.lower() not in {'nan', 'none'}:
                return value
    return str(row.get('grounding_prompt', '')).strip()


def effective_target_object(row):
    for col in ['parsed_target_object', 'target_object', 'target_label', 'target_type']:
        value = str(row.get(col, '')).strip().lower()
        if value and value not in {'nan', 'none', 'object', 'target'}:
            return value
    return 'object'


def target_label_terms(row):
    target = effective_target_object(row)
    target_label = str(row.get('target_label', '')).strip().lower()
    target_type = str(row.get('target_type', '')).strip().lower()
    prompt = model_prompt_text(row).lower()
    candidate_prompt_string = str(row.get('parser_candidate_prompts', '')).strip()

    terms = []
    if candidate_prompt_string:
        terms.extend([x.strip() for x in candidate_prompt_string.split('|') if x.strip()])

    terms += [target_label, target, target_type]

    if target == 'package' or 'package' in prompt or 'box' in prompt or target_type == 'package':
        terms += ['package', 'box', 'cardboard box', 'delivery package', 'parcel']
    if target in {'man', 'woman', 'girl', 'boy', 'person'} or target_type == 'person':
        terms += ['person', 'human']
    if target == 'door' or 'door' in prompt:
        terms += ['door', 'hallway door', 'apartment door']
    if 'button' in target or 'button' in target_type or 'button' in prompt:
        terms += ['elevator button panel', 'button panel', 'elevator button', 'button']
    if 'chair' in target or 'chair' in prompt:
        terms += ['chair']
    if 'table' in target or 'table' in prompt or 'desk' in prompt:
        terms += ['table', 'desk']
    if 'couch' in target or 'sofa' in prompt or 'couch' in prompt:
        terms += ['couch', 'sofa']
    if 'elevator' in target or 'elevator' in prompt:
        terms += ['elevator']
    if target in {'lobby', 'lounge', 'hallway'}:
        terms += [target]

    return unique_keep_order([t for t in terms if t and t not in {'nan', 'none'}])


def relation_label_terms(row):
    prompt = model_prompt_text(row)
    location_hint = str(row.get('parsed_location_hint', '')).strip()
    simple = simplified_prompt_text(prompt)
    target_terms = target_label_terms(row)
    target = target_terms[0] if target_terms else effective_target_object(row)

    terms = []
    if simple:
        terms.append(simple)

    prompt_l = f'{prompt} {location_hint}'.lower()
    if 'elevator' in prompt_l and target:
        terms.append(f'{target} near elevator')
        terms.append(f'{target} by elevator')
    if 'chair' in prompt_l and target:
        terms += [f'{target} beside chair', f'{target} near chair', f'{target} by chair', f'{target} on chair']
    if 'door' in prompt_l and target:
        terms += [f'{target} near door', f'{target} by door']
    if 'shoes' in prompt_l and target:
        terms.append(f'{target} beside shoes')
    if 'books' in prompt_l and target:
        terms.append(f'{target} beside books')
    if 'box' in prompt_l and target and target != 'box':
        terms.append(f'{target} beside box')
    if 'table' in prompt_l and target:
        terms += [f'{target} on table', f'{target} near table']
    if 'floor' in prompt_l and target:
        terms.append(f'{target} on floor')
    if 'couch' in prompt_l or 'sofa' in prompt_l:
        terms += [f'{target} near couch', f'{target} beside couch']

    return unique_keep_order([t for t in terms if t and len(t) <= 80])


def get_grounding_candidates(row):
    target_terms = target_label_terms(row)
    relation_terms = relation_label_terms(row)
    parsed_prompt = str(row.get('model_grounding_prompt', row.get('parsed_grounding_prompt', ''))).strip()
    original_prompt = str(row.get('original_grounding_prompt', row.get('grounding_prompt', ''))).strip()

    if CANDIDATE_LABEL_MODE == 'short':
        candidates = target_terms
    elif CANDIDATE_LABEL_MODE == 'description_first':
        candidates = relation_terms + target_terms
    else:
        # Best default for DINO and OWL ViT: short object labels first, short relation prompts second.
        candidates = target_terms + relation_terms

    if parsed_prompt:
        candidates.insert(0, parsed_prompt)
    if original_prompt and original_prompt != parsed_prompt:
        candidates.append(original_prompt)

    cleaned = unique_keep_order([str(c).strip().lower().rstrip('.') for c in candidates if str(c).strip()])
    return cleaned[:MAX_GROUNDING_CANDIDATES]


MODEL_COLORS = {
    'grounding_dino': 'red',
    'owlvit': 'blue',
    'gpt_vision': 'cyan',
}
GROUND_TRUTH_COLOR = 'lime'


def image_size_from_row(row):
    image_path = PROJECT_ROOT / row['image_path']
    with Image.open(image_path) as img:
        return img.size


def get_ground_truth_box(row):
    raw_size = image_size_from_row(row)
    box = [
        row.get('gt_x_min', np.nan),
        row.get('gt_y_min', np.nan),
        row.get('gt_x_max', np.nan),
        row.get('gt_y_max', np.nan),
    ]
    return normalize_box_xyxy(box, raw_size), raw_size, str(row.get('ground_truth_source', 'corrected_csv'))


def box_area_ratio_xyxy(box, image_size):
    box = normalize_box_xyxy(box, image_size)
    if box is None:
        return 0.0
    x1, y1, x2, y2 = box
    w, h = image_size
    return float(((x2 - x1) * (y2 - y1)) / max(1.0, w * h))


def usable_detection_box(box, image_size):
    norm = normalize_box_xyxy(box, image_size)
    if norm is None:
        return False
    area_ratio = box_area_ratio_xyxy(norm, image_size)
    if area_ratio < MIN_PRED_BOX_AREA_RATIO:
        return False
    if area_ratio > MAX_PRED_BOX_AREA_RATIO:
        return False
    if area_ratio >= FULL_IMAGE_REJECT_RATIO:
        return False
    return True


def detection_rank_score(score, label, labels, box, image_size):
    label = str(label).lower()
    rank = float(score)
    if label in [str(x).lower() for x in labels[:3]]:
        rank += LABEL_PRIORITY_BONUS
    rank -= PRED_BOX_AREA_PENALTY * box_area_ratio_xyxy(box, image_size)
    return rank


def result_pred_box(result, image_size):
    if result is None:
        return None
    box = [
        result.get('pred_x_min', np.nan),
        result.get('pred_y_min', np.nan),
        result.get('pred_x_max', np.nan),
        result.get('pred_y_max', np.nan),
    ]
    return normalize_box_xyxy(box, image_size)


def box_iou_xyxy(a, b):
    if a is None or b is None:
        return 0.0

    ax1, ay1, ax2, ay2 = [float(v) for v in a]
    bx1, by1, bx2, by2 = [float(v) for v in b]

    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)

    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter

    return float(inter / denom) if denom > 0 else 0.0


def grade_iou(iou, status):
    if status != 'success':
        return 'failed'
    if iou >= STRICT_SUCCESS_IOU_THRESHOLD:
        return 'strict_success'
    if iou >= SUCCESS_IOU_THRESHOLD:
        return 'success'
    if iou >= WEAK_OVERLAP_IOU_THRESHOLD:
        return 'weak_overlap'
    return 'poor'



def draw_box(draw, box, color, label, width=8):
    """Draw a very visible box.

    The earlier screenshots could make model boxes look absent when boxes were thin,
    hidden under the top caption, or overwritten by later image saving. This version
    draws thicker outlines, corner ticks, a center crosshair, and a label.
    """
    if box is None:
        return

    try:
        x1, y1, x2, y2 = [int(round(float(v))) for v in box]
    except Exception:
        return

    if x2 <= x1 or y2 <= y1:
        return

    # Main outline, drawn inward/outward for visibility.
    for offset in range(max(1, int(width))):
        draw.rectangle([x1 - offset, y1 - offset, x2 + offset, y2 + offset], outline=color)

    # Corner ticks make boxes easier to see when the predicted region is large.
    tick = max(18, min(60, int(min((x2 - x1), (y2 - y1)) * 0.20)))
    for off in range(max(2, width // 2)):
        # top left
        draw.line([x1, y1 + off, x1 + tick, y1 + off], fill=color, width=3)
        draw.line([x1 + off, y1, x1 + off, y1 + tick], fill=color, width=3)
        # top right
        draw.line([x2 - tick, y1 + off, x2, y1 + off], fill=color, width=3)
        draw.line([x2 - off, y1, x2 - off, y1 + tick], fill=color, width=3)
        # bottom left
        draw.line([x1, y2 - off, x1 + tick, y2 - off], fill=color, width=3)
        draw.line([x1 + off, y2 - tick, x1 + off, y2], fill=color, width=3)
        # bottom right
        draw.line([x2 - tick, y2 - off, x2, y2 - off], fill=color, width=3)
        draw.line([x2 - off, y2 - tick, x2 - off, y2], fill=color, width=3)

    # Center crosshair helps confirm a prediction exists even when the box is large.
    cx = int(round((x1 + x2) / 2))
    cy = int(round((y1 + y2) / 2))
    draw.line([cx - 12, cy, cx + 12, cy], fill=color, width=3)
    draw.line([cx, cy - 12, cx, cy + 12], fill=color, width=3)

    if label:
        text_y = max(38, y1 - 24)
        label_text = str(label)[:70]
        try:
            text_box = draw.textbbox((x1, text_y), label_text)
            label_w = (text_box[2] - text_box[0]) + 10
            label_h = (text_box[3] - text_box[1]) + 8
        except Exception:
            label_w = max(120, min(520, 9 * len(label_text)))
            label_h = 24
        draw.rectangle([x1, text_y, x1 + label_w, text_y + label_h], fill=color)
        text_fill = 'black' if color in {'lime', 'yellow', 'cyan', 'white', 'magenta'} else 'white'
        draw.text((x1 + 5, text_y + 4), label_text, fill=text_fill)


def draw_single_model_screenshots(results_df, model_key, max_images=None):
    out_dir = SCREENSHOT_DIR / model_key
    out_dir.mkdir(parents=True, exist_ok=True)
    results_by_id = {str(r['image_id']): r for _, r in results_df.iterrows()}
    saved = []
    audit_rows = []

    rows = benchmark.to_dict('records')
    if max_images is not None:
        rows = rows[:int(max_images)]

    model_color = MODEL_COLORS.get(model_key, 'red')

    for row in rows:
        image_id = str(row['image_id'])
        image_path = PROJECT_ROOT / row['image_path']
        img = Image.open(image_path).convert('RGB')
        draw = ImageDraw.Draw(img)

        gt_box, raw_size, gt_source = get_ground_truth_box(row)
        pred_row = results_by_id.get(image_id)
        pred_box = result_pred_box(pred_row, raw_size) if pred_row is not None else None
        iou = box_iou_xyxy(pred_box, gt_box)
        status = str(pred_row.get('status', 'missing') if pred_row is not None else 'missing')
        matched_phrase = str(pred_row.get('matched_phrase', '') if pred_row is not None else '')

        # Draw the header first. Then draw boxes on top so boxes are not hidden by the black header.
        draw.rectangle([0, 0, img.width, 38], fill='black')
        draw.text((8, 10), f"{image_id} | {model_key} | status {status} | IoU {iou:.2f}", fill='white')

        draw_box(draw, gt_box, GROUND_TRUTH_COLOR, 'GT', width=6)
        if pred_box is not None:
            draw_box(draw, pred_box, model_color, f"{model_key} pred {iou:.2f}", width=9)
        else:
            draw.rectangle([8, img.height - 34, min(img.width - 8, 650), img.height - 8], fill='black')
            draw.text((14, img.height - 29), f"No valid prediction box for {model_key}", fill=model_color)

        # Bottom caption records the actual coordinates, so it is obvious whether a box exists.
        if pred_box is not None:
            coord_text = f"pred xyxy: {int(pred_box[0])},{int(pred_box[1])},{int(pred_box[2])},{int(pred_box[3])} | phrase: {matched_phrase[:70]}"
        else:
            coord_text = f"pred xyxy: none | phrase: {matched_phrase[:70]}"
        draw.rectangle([0, img.height - 30, img.width, img.height], fill='black')
        draw.text((8, img.height - 22), coord_text, fill='white')

        out_path = out_dir / f'{image_id}.{SCREENSHOT_IMAGE_FORMAT}'
        if SCREENSHOT_IMAGE_FORMAT.lower() == 'png':
            img.save(out_path)
        else:
            img.save(out_path, quality=95)
        saved.append(str(out_path.relative_to(PROJECT_ROOT)).replace('\\', '/'))

        audit_rows.append({
            'model_key': model_key,
            'image_id': image_id,
            'status': status,
            'has_valid_prediction_box': pred_box is not None,
            'iou': iou,
            'pred_x_min': pred_box[0] if pred_box else np.nan,
            'pred_y_min': pred_box[1] if pred_box else np.nan,
            'pred_x_max': pred_box[2] if pred_box else np.nan,
            'pred_y_max': pred_box[3] if pred_box else np.nan,
            'matched_phrase': matched_phrase,
            'screenshot_path': str(out_path.relative_to(PROJECT_ROOT)).replace('\\', '/'),
        })

    audit_path = MODEL_COMPARISON_DIR / f'{model_key}_screenshot_audit.csv'
    pd.DataFrame(audit_rows).to_csv(audit_path, index=False)
    valid_count = int(pd.DataFrame(audit_rows)['has_valid_prediction_box'].sum()) if audit_rows else 0
    print(f'Saved {model_key} screenshots: {len(saved)} | with visible prediction boxes: {valid_count}')
    print(f'Saved screenshot audit: {audit_path}')

    return saved


def finalize_model_results(pred_rows, model_name, model_key):
    pred_df = pd.DataFrame(pred_rows)
    if pred_df.empty:
        pred_df = pd.DataFrame(columns=['image_id', 'status'])

    # Make expected columns exist, so failed rows and success rows have the same schema.
    for col in ['image_id', 'status', 'pred_x_min', 'pred_y_min', 'pred_x_max', 'pred_y_max', 'confidence', 'matched_phrase', 'notes']:
        if col not in pred_df.columns:
            pred_df[col] = np.nan if col.startswith('pred_') or col == 'confidence' else ''

    pred_df['image_id'] = pred_df['image_id'].astype(str)
    by_id = {str(r['image_id']): r for _, r in pred_df.iterrows()}

    final_rows = []

    for row in benchmark.to_dict('records'):
        image_id = str(row['image_id'])
        pred = dict(by_id.get(image_id, {'image_id': image_id, 'status': 'failed', 'notes': 'no prediction row'}))
        gt_box, raw_size, gt_source = get_ground_truth_box(row)
        pred_box = result_pred_box(pred, raw_size)

        status = str(pred.get('status', 'failed')).lower()
        if pred_box is None:
            status = 'failed'

        iou = box_iou_xyxy(pred_box, gt_box)
        quality = grade_iou(iou, status)

        final_rows.append({
            'model_name': model_name,
            'model_key': model_key,
            'image_id': image_id,
            'image_path': row.get('image_path', ''),
            'grounding_prompt': row.get('grounding_prompt', ''),
            'source_command': row.get('source_command', ''),
            'parsed_action': row.get('parsed_action', ''),
            'parsed_target_object': row.get('parsed_target_object', ''),
            'parsed_location_hint': row.get('parsed_location_hint', ''),
            'model_grounding_prompt': row.get('model_grounding_prompt', row.get('grounding_prompt', '')),
            'original_grounding_prompt': row.get('original_grounding_prompt', ''),
            'target_object': row.get('target_object', ''),
            'target_type': row.get('target_type', ''),
            'gt_x_min': gt_box[0] if gt_box else np.nan,
            'gt_y_min': gt_box[1] if gt_box else np.nan,
            'gt_x_max': gt_box[2] if gt_box else np.nan,
            'gt_y_max': gt_box[3] if gt_box else np.nan,
            'ground_truth_source': gt_source,
            'pred_x_min': pred_box[0] if pred_box else np.nan,
            'pred_y_min': pred_box[1] if pred_box else np.nan,
            'pred_x_max': pred_box[2] if pred_box else np.nan,
            'pred_y_max': pred_box[3] if pred_box else np.nan,
            'confidence': pred.get('confidence', np.nan),
            'matched_phrase': pred.get('matched_phrase', ''),
            'status': status,
            'has_valid_prediction_box': bool(pred_box is not None),
            'iou': iou,
            'target_correct': bool(iou >= SUCCESS_IOU_THRESHOLD and status == 'success'),
            'bbox_quality': quality,
            'notes': pred.get('notes', ''),
            'box_source': pred.get('box_source', ''),
            'sam_mask_score': pred.get('sam_mask_score', np.nan),
            'sam_mask_index': pred.get('sam_mask_index', np.nan),
            'sam_component_pixels': pred.get('sam_component_pixels', np.nan),
            'sam_component_area_ratio': pred.get('sam_component_area_ratio', np.nan),
            'sam_prompt_box_iou': pred.get('sam_prompt_box_iou', np.nan),
            'dino_pred_x_min': pred.get('dino_pred_x_min', np.nan),
            'dino_pred_y_min': pred.get('dino_pred_y_min', np.nan),
            'dino_pred_x_max': pred.get('dino_pred_x_max', np.nan),
            'dino_pred_y_max': pred.get('dino_pred_y_max', np.nan),
        })

    results_df = pd.DataFrame(final_rows)
    results_path = OUTPUT_DIR / f'{model_key}_results.csv'
    results_df.to_csv(results_path, index=False)

    draw_single_model_screenshots(results_df, model_key, MAX_SCREENSHOTS_PER_MODEL)

    metrics = {
        'model_name': model_name,
        'model_key': model_key,
        'rows': len(results_df),
        'successful_predictions': int((results_df['status'] == 'success').sum()),
        'valid_prediction_boxes': int(results_df['has_valid_prediction_box'].sum()),
        'mean_iou': float(results_df['iou'].mean()),
        'median_iou': float(results_df['iou'].median()),
        'primary_success_threshold': SUCCESS_IOU_THRESHOLD,
        'strict_success_threshold': STRICT_SUCCESS_IOU_THRESHOLD,
        'success_count_primary': int((results_df['iou'] >= SUCCESS_IOU_THRESHOLD).sum()),
        'success_count_iou_050_strict': int((results_df['iou'] >= STRICT_SUCCESS_IOU_THRESHOLD).sum()),
        'weak_or_better_count_iou_010': int((results_df['iou'] >= WEAK_OVERLAP_IOU_THRESHOLD).sum()),
        'success_rate_primary': float((results_df['iou'] >= SUCCESS_IOU_THRESHOLD).mean()),
        'success_rate_iou_050_strict': float((results_df['iou'] >= STRICT_SUCCESS_IOU_THRESHOLD).mean()),
        'success_rate_iou_025': float((results_df['iou'] >= 0.25).mean()),
        'weak_or_better_rate_iou_010': float((results_df['iou'] >= WEAK_OVERLAP_IOU_THRESHOLD).mean()),
    }

    metrics_path = MODEL_COMPARISON_DIR / f'{model_key}_metrics.csv'
    pd.DataFrame([metrics]).to_csv(metrics_path, index=False)

    print(f'Saved results: {results_path}')
    print(f'Saved metrics: {metrics_path}')
    display(pd.DataFrame([metrics]))
    return results_df, metrics


def refresh_saved_result_metrics():
    result_files = sorted(OUTPUT_DIR.glob('*_results.csv'))
    if not result_files:
        print('No saved model result files found to refresh.')
        return []

    refreshed = []
    for path in result_files:
        model_key = path.name.replace('_results.csv', '')
        df = pd.read_csv(path)
        if 'image_id' not in df.columns:
            continue

        model_name = str(df['model_name'].iloc[0]) if 'model_name' in df.columns and len(df) else model_key
        pred_rows = []

        for _, r in df.iterrows():
            pred_rows.append({
                'image_id': r.get('image_id', ''),
                'pred_x_min': r.get('pred_x_min', np.nan),
                'pred_y_min': r.get('pred_y_min', np.nan),
                'pred_x_max': r.get('pred_x_max', np.nan),
                'pred_y_max': r.get('pred_y_max', np.nan),
                'confidence': r.get('confidence', np.nan),
                'matched_phrase': r.get('matched_phrase', ''),
                'status': r.get('status', 'success'),
                'notes': r.get('notes', ''),
                'box_source': r.get('box_source', ''),
                'sam_mask_score': r.get('sam_mask_score', np.nan),
                'sam_mask_index': r.get('sam_mask_index', np.nan),
                'sam_component_pixels': r.get('sam_component_pixels', np.nan),
                'sam_component_area_ratio': r.get('sam_component_area_ratio', np.nan),
                'sam_prompt_box_iou': r.get('sam_prompt_box_iou', np.nan),
                'dino_pred_x_min': r.get('dino_pred_x_min', np.nan),
                'dino_pred_y_min': r.get('dino_pred_y_min', np.nan),
                'dino_pred_x_max': r.get('dino_pred_x_max', np.nan),
                'dino_pred_y_max': r.get('dino_pred_y_max', np.nan),
            })

        finalize_model_results(pred_rows, model_name, model_key)
        refreshed.append(model_key)

    print('Refreshed saved results:', refreshed)
    return refreshed



## Grounding DINO Baseline

This section runs Grounding DINO through the Hugging Face Transformers loader. This avoids the official repo editable install, which often fails in Colab because it tries to build native CUDA/C++ extensions from a Google Drive folder.


In [ ]:

def setup_grounding_dino():
    import torch
    from transformers import pipeline

    model_id = 'IDEA-Research/grounding-dino-tiny'
    device_id = 0 if DEVICE == 'cuda' else -1
    print(f'Loading {model_id} through Transformers pipeline...')
    detector = pipeline(
        task='zero-shot-object-detection',
        model=model_id,
        device=device_id,
    )
    return detector, model_id


def detector_predict_best(detector, image, candidates, thresholds):
    best = None
    labels = [c.rstrip('.').strip() for c in candidates if str(c).strip()]
    labels = unique_keep_order(labels)
    image_size = image.size

    for threshold in thresholds:
        try:
            preds = detector(image, candidate_labels=labels, threshold=float(threshold))
        except TypeError:
            preds = detector(image, labels, threshold=float(threshold))

        if not preds:
            continue
        for pred in preds:
            score = float(pred.get('score', 0.0))
            box = pred.get('box', {})
            if not box:
                continue
            box_xyxy = [box.get('xmin'), box.get('ymin'), box.get('xmax'), box.get('ymax')]
            if not usable_detection_box(box_xyxy, image_size):
                continue
            label = pred.get('label', '')
            rank_score = detection_rank_score(score, label, labels, box_xyxy, image_size)
            candidate = {
                'box': box_xyxy,
                'score': score,
                'rank_score': rank_score,
                'label': label,
                'threshold': threshold,
                'area_ratio': box_area_ratio_xyxy(box_xyxy, image_size),
            }
            if best is None or rank_score > best['rank_score']:
                best = candidate
        if best is not None:
            return best
    return best


def run_grounding_dino():
    detector, model_id = setup_grounding_dino()
    pred_rows = []

    for row in tqdm(benchmark.to_dict('records'), desc='Grounding DINO'):
        image_path = PROJECT_ROOT / row['image_path']
        try:
            image = Image.open(image_path).convert('RGB')
            candidates = get_grounding_candidates(row)
            thresholds = [float(t) for t in GROUNDING_DINO_RETRY_THRESHOLDS]
            best = detector_predict_best(detector, image, candidates, thresholds)

            if best is None:
                pred_rows.append({
                    'image_id': row['image_id'],
                    'status': 'failed',
                    'confidence': np.nan,
                    'matched_phrase': '',
                    'notes': 'pipeline returned no boxes for: ' + ' | '.join(candidates[:6]),
                })
                continue

            pred_box = normalize_box_xyxy(best['box'], image.size)
            if pred_box is None:
                pred_rows.append({'image_id': row['image_id'], 'status': 'failed', 'confidence': best['score'], 'matched_phrase': best['label'], 'notes': 'invalid DINO box after normalization'})
                continue
            x1, y1, x2, y2 = pred_box
            pred_rows.append({
                'image_id': row['image_id'],
                'pred_x_min': x1,
                'pred_y_min': y1,
                'pred_x_max': x2,
                'pred_y_max': y2,
                'confidence': best['score'],
                'matched_phrase': best['label'],
                'status': 'success',
                'notes': f"{model_id}; threshold={best['threshold']}; area={best['area_ratio']:.3f}; rank={best['rank_score']:.3f}",
            })
        except Exception as exc:
            pred_rows.append({
                'image_id': row['image_id'],
                'status': 'failed',
                'confidence': np.nan,
                'matched_phrase': '',
                'notes': str(exc)[:500],
            })

    results_df, metrics = finalize_model_results(pred_rows, 'Grounding DINO', 'grounding_dino')
    if int((results_df['status'] == 'success').sum()) == 0:
        display(results_df[['image_id', 'notes']].head(10))
    return results_df, metrics

if RUN_GROUNDING_DINO:
    grounding_dino_results, grounding_dino_metrics = run_grounding_dino()
else:
    print('Grounding DINO skipped.')


## OWL ViT Comparison

This section runs OWL ViT with the same prompts and benchmark rows. It is a useful comparison point, even though it may be weaker on spatial phrases than Grounding DINO.

In [ ]:

def setup_owlvit():
    import torch
    from transformers import pipeline

    model_id = 'google/owlvit-base-patch32'
    device_id = 0 if DEVICE == 'cuda' else -1
    print(f'Loading {model_id} through Transformers pipeline...')
    detector = pipeline(
        task='zero-shot-object-detection',
        model=model_id,
        device=device_id,
    )
    return detector, model_id


def run_owlvit():
    detector, model_id = setup_owlvit()
    pred_rows = []

    for row in tqdm(benchmark.to_dict('records'), desc='OWL ViT'):
        image_path = PROJECT_ROOT / row['image_path']
        try:
            image = Image.open(image_path).convert('RGB')
            candidates = get_grounding_candidates(row)
            best = detector_predict_best(detector, image, candidates, OWL_VIT_RETRY_THRESHOLDS)

            if best is None:
                pred_rows.append({
                    'image_id': row['image_id'],
                    'status': 'failed',
                    'confidence': np.nan,
                    'matched_phrase': '',
                    'notes': 'pipeline returned no boxes for: ' + ' | '.join(candidates[:6]),
                })
                continue

            x1, y1, x2, y2 = best['box']
            pred_rows.append({
                'image_id': row['image_id'],
                'pred_x_min': x1,
                'pred_y_min': y1,
                'pred_x_max': x2,
                'pred_y_max': y2,
                'confidence': best['score'],
                'matched_phrase': best['label'],
                'status': 'success',
                'notes': f"{model_id}; threshold={best['threshold']}; area={best['area_ratio']:.3f}; rank={best['rank_score']:.3f}",
            })
        except Exception as exc:
            pred_rows.append({
                'image_id': row['image_id'],
                'status': 'failed',
                'confidence': np.nan,
                'matched_phrase': '',
                'notes': str(exc)[:500],
            })

    results_df, metrics = finalize_model_results(pred_rows, 'OWL ViT', 'owlvit')
    if int((results_df['status'] == 'success').sum()) == 0:
        display(results_df[['image_id', 'notes']].head(10))
    return results_df, metrics

if RUN_OWL_VIT:
    owlvit_results, owlvit_metrics = run_owlvit()
else:
    print('OWL ViT skipped.')


## GPT Vision Interpretation

This section uses the same plain JSON request pattern that worked in the earlier notebook version. It writes results to the same comparison format as the local grounding models.


In [ ]:
def extract_json_object(text):
    text = str(text).strip()
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError('No JSON object found in model response.')
    return json.loads(text[start:end + 1])


def image_to_data_url(image_path):
    import base64
    suffix = Path(image_path).suffix.lower()
    mime = 'image/png' if suffix == '.png' else 'image/jpeg'
    data = base64.b64encode(Path(image_path).read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{data}'


def parsed_box_has_coordinates(parsed):
    keys = ['x_min', 'y_min', 'x_max', 'y_max']
    try:
        vals = [float(parsed.get(k)) for k in keys]
    except Exception:
        return False
    return all(np.isfinite(vals))


def run_gpt_vision():
    from openai import OpenAI
    from google.colab import userdata

    if not os.environ.get('OPENAI_API_KEY'):
        try:
            api_key = userdata.get('OPENAI_API_KEY')
            os.environ['OPENAI_API_KEY'] = api_key
            print('OPENAI_API_KEY retrieved from Colab secrets.')
        except userdata.SecretNotFoundError:
            raise EnvironmentError('OPENAI_API_KEY not found in Colab secrets. Set it as OPENAI_API_KEY.')
        except Exception as exc:
            raise EnvironmentError(f'Failed to read OPENAI_API_KEY from Colab secrets: {exc}')

    if not os.environ.get('OPENAI_API_KEY'):
        raise EnvironmentError('OPENAI_API_KEY is empty.')

    client = OpenAI()
    pred_rows = []
    rows = benchmark.to_dict('records')
    if GPT_VISION_MAX_ROWS is not None:
        rows = rows[:int(GPT_VISION_MAX_ROWS)]

    for row in tqdm(rows, desc='GPT vision'):
        image_path = PROJECT_ROOT / row['image_path']
        source_command = str(row.get('source_command', row.get('command', row.get('grounding_prompt', '')))).strip()
        parsed_target = str(row.get('parsed_target_object', row.get('target_object', 'target object'))).strip()
        parsed_location = str(row.get('parsed_location_hint', '')).strip()
        prompt = clean_prompt(row.get('model_grounding_prompt', row.get('parsed_grounding_prompt', row.get('grounding_prompt', ''))))
        try:
            img = Image.open(image_path).convert('RGB')
            width, height = img.size
            data_url = image_to_data_url(image_path)
            instruction = f'''
Return one JSON object only.
Image size: width={width}, height={height}.
Original command or description: {source_command}
Parsed target_object: {parsed_target}
Parsed location_hint: {parsed_location}
Model grounding prompt: {prompt}
Find the parsed visible target and return an approximate bounding box in pixel coordinates.
Schema:
{{
  "status": "success" or "failed",
  "x_min": number,
  "y_min": number,
  "x_max": number,
  "y_max": number,
  "confidence": number between 0 and 1,
  "matched_phrase": string,
  "notes": string
}}
'''.strip()

            response = client.responses.create(
                model=GPT_VISION_MODEL,
                input=[
                    {
                        'role': 'user',
                        'content': [
                            {'type': 'input_text', 'text': instruction},
                            {'type': 'input_image', 'image_url': data_url, 'detail': GPT_VISION_DETAIL},
                        ],
                    }
                ],
            )

            parsed = extract_json_object(response.output_text)
            status = str(parsed.get('status', 'failed')).lower().strip()
            raw_box = [parsed.get('x_min'), parsed.get('y_min'), parsed.get('x_max'), parsed.get('y_max')]
            pred_box = normalize_box_xyxy(raw_box, (width, height))
            has_box = pred_box is not None and (status == 'success' or parsed_box_has_coordinates(parsed))

            if has_box:
                pred_rows.append({
                    'image_id': row['image_id'],
                    'pred_x_min': pred_box[0],
                    'pred_y_min': pred_box[1],
                    'pred_x_max': pred_box[2],
                    'pred_y_max': pred_box[3],
                    'confidence': float(parsed.get('confidence', np.nan)) if parsed.get('confidence') is not None else np.nan,
                    'matched_phrase': parsed.get('matched_phrase', ''),
                    'status': 'success',
                    'notes': parsed.get('notes', ''),
                })
            else:
                pred_rows.append({
                    'image_id': row['image_id'],
                    'status': 'failed',
                    'confidence': float(parsed.get('confidence', np.nan)) if parsed.get('confidence') is not None else np.nan,
                    'matched_phrase': parsed.get('matched_phrase', ''),
                    'notes': parsed.get('notes', 'target not found'),
                })
        except Exception as exc:
            pred_rows.append({
                'image_id': row['image_id'],
                'status': 'failed',
                'confidence': np.nan,
                'matched_phrase': '',
                'notes': str(exc)[:500],
            })

    results_df, metrics = finalize_model_results(pred_rows, 'GPT vision interpretation', 'gpt_vision')
    if int((results_df['status'] == 'success').sum()) == 0:
        display(results_df[['image_id', 'notes']].head(GPT_VISION_DEBUG_FAILURES))
    return results_df, metrics

if RUN_GPT_VISION:
    gpt_vision_results, gpt_vision_metrics = run_gpt_vision()
else:
    print('GPT vision interpretation skipped.')


## Combined Metrics And Graphs

Run this section after one or more model sections finish. It reloads saved result CSV files, filters everything to the approved benchmark image IDs, refreshes IoU using the corrected CSV ground truth boxes, and redraws comparison screenshots. The primary success metric uses `SUCCESS_IOU_THRESHOLD = 0.25`; the stricter IoU 0.50 score is still saved as a separate reference metric.


In [ ]:

def load_result_files():
    result_tables = {}
    for path in [path for path in MODEL_RESULT_PATHS.values() if path.exists()]:
        model_key = path.name.replace('_results.csv', '')
        df = pd.read_csv(path)
        if 'image_id' in df.columns:
            result_tables[model_key] = df
    return result_tables


def draw_combined_model_screenshots(max_images=100):
    result_tables = load_result_files()
    if not result_tables:
        print('No model result files found for combined screenshots.')
        return []

    combined_dir = SCREENSHOT_DIR / 'combined_models'
    combined_dir.mkdir(parents=True, exist_ok=True)
    saved = []

    rows = benchmark.to_dict('records')
    if max_images is not None:
        rows = rows[:int(max_images)]

    for row in rows:
        image_path = PROJECT_ROOT / row['image_path']
        img = Image.open(image_path).convert('RGB')
        draw = ImageDraw.Draw(img)

        gt_box, raw_size, gt_source = get_ground_truth_box(row)
        draw.rectangle([0, 0, img.width, 38], fill='black')
        draw.text((8, 10), f"{row['image_id']} | combined predictions | GT: {gt_source}", fill='white')

        draw_box(draw, gt_box, GROUND_TRUTH_COLOR, 'GT', width=6)

        legend_y = 44
        for model_key, df in result_tables.items():
            match = df[df['image_id'].astype(str) == str(row['image_id'])]
            if match.empty:
                continue

            result = match.iloc[0].to_dict()
            pred_box = result_pred_box(result, raw_size)
            if pred_box is None:
                continue

            color = MODEL_COLORS.get(model_key, 'red')
            model_name = str(result.get('model_name', model_key))
            iou = box_iou_xyxy(pred_box, gt_box)

            draw_box(draw, pred_box, color, f"{model_name} {iou:.2f}", width=8)
            draw.rectangle([8, legend_y, 26, legend_y + 14], fill=color)
            draw.text((34, legend_y - 1), f"{model_name}: IoU {iou:.2f}", fill='white')
            legend_y += 20

        out_path = combined_dir / f"{row['image_id']}.{SCREENSHOT_IMAGE_FORMAT}"
        if SCREENSHOT_IMAGE_FORMAT.lower() == 'png':
            img.save(out_path)
        else:
            img.save(out_path, quality=95)
        saved.append(str(out_path.relative_to(PROJECT_ROOT)).replace('\\', '/'))

    manifest_path = combined_dir / 'combined_screenshots_manifest.csv'
    pd.DataFrame({'combined_screenshot_path': saved}).to_csv(manifest_path, index=False)
    print(f'Saved combined screenshots: {len(saved)}')
    print(f'Saved manifest: {manifest_path}')
    return saved


def plot_model_comparison(refresh_existing=True):
    if refresh_existing:
        refresh_saved_result_metrics()

    result_files = [path for path in MODEL_RESULT_PATHS.values() if path.exists()]
    if not result_files:
        print('No result files found yet.')
        return pd.DataFrame()

    summary_rows = []
    grade_rows = []

    for path in result_files:
        df = pd.read_csv(path)
        if 'iou' not in df.columns or 'model_name' not in df.columns:
            continue

        model_name = str(df['model_name'].iloc[0])
        summary_rows.append({
            'model_name': model_name,
            'result_file': str(path.relative_to(PROJECT_ROOT)).replace('\\', '/'),
            'rows': len(df),
            'mean_iou': df['iou'].mean(),
            'median_iou': df['iou'].median(),
            'primary_success_threshold': SUCCESS_IOU_THRESHOLD,
            'success_rate_primary': (df['iou'] >= SUCCESS_IOU_THRESHOLD).mean(),
            'success_rate_iou_050_strict': (df['iou'] >= STRICT_SUCCESS_IOU_THRESHOLD).mean(),
            'success_rate_iou_025': (df['iou'] >= 0.25).mean(),
            'weak_or_better_rate_iou_010': (df['iou'] >= WEAK_OVERLAP_IOU_THRESHOLD).mean(),
            'failed_rate': (df['status'] != 'success').mean(),
        })

        counts = df['bbox_quality'].value_counts().to_dict()
        for grade in ['strict_success', 'success', 'weak_overlap', 'poor', 'failed']:
            grade_rows.append({'model_name': model_name, 'bbox_quality': grade, 'count': counts.get(grade, 0)})

    summary = pd.DataFrame(summary_rows)
    summary_path = MODEL_COMPARISON_DIR / 'all_model_metrics.csv'
    summary.to_csv(summary_path, index=False)

    if summary.empty:
        print('No usable result files found yet.')
        return summary

    plt.figure(figsize=(8, 4))
    plt.bar(summary['model_name'], summary['mean_iou'])
    plt.ylabel('Mean IoU')
    plt.ylim(0, 1)
    plt.xticks(rotation=20, ha='right')
    plt.title('Mean IoU by model')
    plt.tight_layout()
    plt.savefig(MODEL_COMPARISON_DIR / 'mean_iou_by_model.png', dpi=180)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.bar(summary['model_name'], summary['success_rate_primary'])
    plt.ylabel(f'Primary success rate, IoU >= {SUCCESS_IOU_THRESHOLD:.2f}')
    plt.ylim(0, 1)
    plt.xticks(rotation=20, ha='right')
    plt.title('Grounding success rate by model')
    plt.tight_layout()
    plt.savefig(MODEL_COMPARISON_DIR / 'success_rate_by_model.png', dpi=180)
    plt.show()

    grades = pd.DataFrame(grade_rows)
    pivot = grades.pivot_table(index='model_name', columns='bbox_quality', values='count', fill_value=0)
    for col in ['strict_success', 'success', 'weak_overlap', 'poor', 'failed']:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot = pivot[['strict_success', 'success', 'weak_overlap', 'poor', 'failed']]
    pivot.to_csv(MODEL_COMPARISON_DIR / 'bbox_quality_counts_by_model.csv')
    pivot.plot(kind='bar', stacked=True, figsize=(9, 5))
    plt.ylabel('Frame count')
    plt.title('Box quality counts by model')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(MODEL_COMPARISON_DIR / 'bbox_quality_counts_by_model.png', dpi=180)
    plt.show()

    target_rows = []
    for path in result_files:
        model_df = pd.read_csv(path)
        if 'target_object' not in model_df.columns:
            continue
        model_name = str(model_df['model_name'].iloc[0])
        grouped = model_df.groupby('target_object').agg(
            rows=('image_id', 'count'),
            mean_iou=('iou', 'mean'),
            success_rate_primary=('iou', lambda s: (s >= SUCCESS_IOU_THRESHOLD).mean()),
            success_rate_iou_050_strict=('iou', lambda s: (s >= STRICT_SUCCESS_IOU_THRESHOLD).mean()),
            success_rate_iou_025=('iou', lambda s: (s >= 0.25).mean()),
            weak_or_better_rate_iou_010=('iou', lambda s: (s >= WEAK_OVERLAP_IOU_THRESHOLD).mean()),
        ).reset_index()
        grouped.insert(0, 'model_name', model_name)
        target_rows.append(grouped)

    if target_rows:
        target_summary = pd.concat(target_rows, ignore_index=True)
        target_summary.to_csv(MODEL_COMPARISON_DIR / 'metrics_by_target_object.csv', index=False)

        pivot_target = target_summary.pivot_table(index='target_object', columns='model_name', values='success_rate_primary', fill_value=0)
        pivot_target.plot(kind='bar', figsize=(10, 5))
        plt.ylabel(f'Primary success rate, IoU >= {SUCCESS_IOU_THRESHOLD:.2f}')
        plt.ylim(0, 1)
        plt.title('Success rate by target object')
        plt.xticks(rotation=25, ha='right')
        plt.tight_layout()
        plt.savefig(MODEL_COMPARISON_DIR / 'success_rate_by_target_object.png', dpi=180)
        plt.show()

    draw_combined_model_screenshots(MAX_SCREENSHOTS_PER_MODEL)

    print(f'Saved combined metrics: {summary_path}')
    display(summary)
    return summary



In [ ]:
combined_metrics = plot_model_comparison(refresh_existing=True)

print('Output folders:')
print('Model outputs:', OUTPUT_DIR)
print('Screenshots:', SCREENSHOT_DIR)
print('Combined screenshots:', SCREENSHOT_DIR / 'combined_models')
print('Metrics and graphs:', MODEL_COMPARISON_DIR)



# Notebook 03 output contract for Notebooks 04 and 05.
if not PARSED_BENCHMARK_PATH.exists():
    raise FileNotFoundError(f'Parsed benchmark was not created: {PARSED_BENCHMARK_PATH}')

expected_ids = set(benchmark['image_id'].astype(str))
contract_rows = []

for model_key, result_path in MODEL_RESULT_PATHS.items():
    exists = result_path.exists()
    result_ids = set()
    duplicate_count = 0

    if exists:
        result_df = pd.read_csv(result_path)
        if 'image_id' not in result_df.columns:
            raise ValueError(f'{result_path} is missing image_id')
        duplicate_count = int(result_df['image_id'].astype(str).duplicated().sum())
        result_ids = set(result_df['image_id'].astype(str))

        if duplicate_count:
            raise ValueError(f'{result_path} contains duplicate image IDs.')
        if result_ids != expected_ids:
            raise ValueError(
                f'{result_path.name} does not match the current benchmark. '
                f'Missing={sorted(expected_ids - result_ids)[:10]}, '
                f'extra={sorted(result_ids - expected_ids)[:10]}'
            )

    contract_rows.append({
        'model_key': model_key,
        'run_enabled': bool(MODEL_RUN_FLAGS[model_key]),
        'result_path': str(result_path.relative_to(PROJECT_ROOT)).replace('\\', '/'),
        'result_exists': exists,
        'row_count': len(result_ids) if exists else 0,
        'expected_row_count': len(expected_ids),
        'duplicate_image_ids': duplicate_count,
        'id_set_matches_benchmark': bool(exists and result_ids == expected_ids),
    })

contract_df = pd.DataFrame(contract_rows)
contract_path = MODEL_COMPARISON_DIR / 'notebook03_output_contract.csv'
contract_df.to_csv(contract_path, index=False)

missing_enabled = contract_df[
    contract_df['run_enabled'] & ~contract_df['result_exists']
]
if not missing_enabled.empty:
    raise FileNotFoundError(
        'A model was enabled but its result CSV was not created:\n'
        + missing_enabled[['model_key', 'result_path']].to_string(index=False)
    )

print('Notebook 03 integration contract passed for every available result file.')
print('Notebook 04 reads:', PARSED_BENCHMARK_PATH)
print('Notebook 05 reads baseline results from:', OUTPUT_DIR)
display(contract_df)
